# 03 — Exploratory Data Analysis & Visualisation
### UPI Transaction Trends India 2024

**What this notebook does:**
Produces 8 publication-quality charts that answer the core business questions
about India's UPI ecosystem in 2024. Every chart is saved as a PNG for use
in the Power BI dashboard and GitHub README.

| Chart | Business Question |
|---|---|
| 1. Monthly Volume & Value | How did UPI grow through 2024? |
| 2. App Market Share | Who dominates — PhonePe vs Google Pay? |
| 3. P2P vs P2M by Month | Is merchant payment adoption growing? |
| 4. Hourly Heatmap | When are Indians transacting most? |
| 5. Top 10 Cities | Which cities drive UPI volume? |
| 6. Amount Bracket Distribution | Does our data match NPCI's 86% under ₹500 stat? |
| 7. Merchant Category Breakdown | Where is money being spent? |
| 8. Month-over-Month Growth Rate | Which months saw spikes or dips? |


## 1. Imports & Global Style

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Global chart style ────────────────────────────────────────────────────────
# Clean, professional style suitable for a portfolio / dashboard
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F8F8F8',
    'axes.grid':         True,
    'grid.color':        'white',
    'grid.linewidth':    1.2,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.spines.left':  False,
    'axes.spines.bottom':False,
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
})

# Colour palette — consistent across all charts
APP_COLORS = {
    'Phonepe':    '#5C2D91',   # PhonePe purple
    'Google Pay': '#4285F4',   # Google blue
    'Paytm':      '#00B9F1',   # Paytm light blue
    'Bhim':       '#FF6B35',   # BHIM orange
    'Amazon Pay': '#FF9900',   # Amazon yellow
    'Others':     '#94A3B8',   # Neutral grey
}
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
MONTH_SHORT = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

CHARTS_DIR = '../data/processed/charts'
os.makedirs(CHARTS_DIR, exist_ok=True)

print("Setup complete ✓")
print(f"Charts will be saved to: {CHARTS_DIR}")


## 2. Load Data

In [ ]:
df     = pd.read_csv('../data/processed/transactions_clean.csv')
ms     = pd.read_csv('../data/processed/monthly_summary.csv')
npci   = pd.read_csv('../data/raw/npci_monthly_2024.csv')

# Filter out outliers for all charts
df_clean = df[df['is_outlier'] == False].copy()

# Ensure month order is correct in monthly_summary
ms = ms.sort_values('month').reset_index(drop=True)
ms['month_short'] = MONTH_SHORT

print(f"transactions_clean : {len(df_clean):,} rows (outliers excluded)")
print(f"monthly_summary    : {len(ms)} rows")
print(f"npci_monthly       : {len(npci)} rows")


## Chart 1 — Monthly Transaction Volume & Value (Dual Axis)
**Story:** UPI showed consistent growth through 2024 with clear spikes in
festive months (October, December). Volume and value don't always move together —
December had the highest value per transaction.


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))

x     = range(len(ms))
bars  = ax1.bar(x, ms['total_transactions'], color='#5C2D91', alpha=0.75,
                width=0.6, label='Transaction Count (Synthetic)')
ax1.set_ylabel('Transaction Count', color='#5C2D91', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#5C2D91')
ax1.set_xticks(x)
ax1.set_xticklabels(MONTH_SHORT)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{int(v):,}'))

# Overlay: NPCI real volume (scaled to fit axis)
scale = ms['total_transactions'].max() / npci['volume_million'].max()
ax1.plot(x, npci['volume_million'] * scale, color='#FF6B35', linewidth=2,
         linestyle='--', marker='o', markersize=5, label='NPCI Real Volume (scaled)')

ax2 = ax1.twinx()
ax2.plot(x, ms['total_value_cr'], color='#0EA5E9', linewidth=2.5,
         marker='s', markersize=6, label='Total Value (₹ Cr, Synthetic)')
ax2.set_ylabel('Total Value (₹ Crore)', color='#0EA5E9', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#0EA5E9')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'₹{v:,.0f}Cr'))

# Add value labels on bars
for bar, val in zip(bars, ms['total_transactions']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{val:,}', ha='center', va='bottom', fontsize=7.5, color='#5C2D91')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9,
           framealpha=0.9)

ax1.set_title('Monthly UPI Transaction Volume & Value — 2024', pad=12)
fig.text(0.5, -0.02,
         'Bars = synthetic transaction count  |  Orange dashed = real NPCI volume (scaled)  |  Blue line = synthetic value',
         ha='center', fontsize=8.5, color='#64748B')

plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/01_monthly_volume_value.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 1 saved ✓")


## Chart 2 — UPI App Market Share
**Story:** PhonePe dominates with ~48% volume share, Google Pay second at ~37%.
Together they account for 85% of all transactions — mirroring real 2024 market data.


In [ ]:
app_stats = (
    df_clean.groupby('upi_app')
    .agg(txn_count=('transaction_id','count'),
         total_value=('amount_inr','sum'))
    .reset_index()
    .sort_values('txn_count', ascending=True)   # ascending for horizontal bar
)
app_stats['volume_pct'] = app_stats['txn_count'] / app_stats['txn_count'].sum() * 100
app_stats['value_pct']  = app_stats['total_value'] / app_stats['total_value'].sum() * 100
colors = [APP_COLORS.get(a, '#94A3B8') for a in app_stats['upi_app']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: volume share
bars1 = axes[0].barh(app_stats['upi_app'], app_stats['volume_pct'],
                     color=colors, height=0.55, edgecolor='white')
axes[0].set_xlabel('Share of Total Transactions (%)')
axes[0].set_title('Market Share by Transaction Volume')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
for bar, val in zip(bars1, app_stats['volume_pct']):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
axes[0].set_xlim(0, app_stats['volume_pct'].max() * 1.18)

# Right: value share
bars2 = axes[1].barh(app_stats['upi_app'], app_stats['value_pct'],
                     color=colors, height=0.55, edgecolor='white')
axes[1].set_xlabel('Share of Total Transaction Value (%)')
axes[1].set_title('Market Share by Transaction Value')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
for bar, val in zip(bars2, app_stats['value_pct']):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlim(0, app_stats['value_pct'].max() * 1.18)

fig.suptitle('UPI App Market Share — 2024', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/02_app_market_share.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 2 saved ✓")


## Chart 3 — P2P vs P2M Split by Month (Stacked Bar)
**Story:** P2M (merchant payments) consistently grows its share quarter on quarter,
reflecting India's shift from person-to-person transfers toward everyday retail spending.


In [ ]:
monthly_type = (
    df_clean.groupby(['month', 'month_name', 'transaction_type'])
    .size().reset_index(name='count')
    .pivot(index=['month','month_name'], columns='transaction_type', values='count')
    .reset_index()
    .sort_values('month')
)
monthly_type['total'] = monthly_type['P2P'] + monthly_type['P2M']
monthly_type['p2p_pct'] = monthly_type['P2P'] / monthly_type['total'] * 100
monthly_type['p2m_pct'] = monthly_type['P2M'] / monthly_type['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = range(12)

# Left: absolute counts stacked
axes[0].bar(x, monthly_type['P2P'], label='P2P', color='#5C2D91', alpha=0.85, width=0.6)
axes[0].bar(x, monthly_type['P2M'], bottom=monthly_type['P2P'],
            label='P2M', color='#0EA5E9', alpha=0.85, width=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MONTH_SHORT)
axes[0].set_ylabel('Number of Transactions')
axes[0].set_title('P2P vs P2M — Absolute Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
axes[0].legend(loc='upper left')

# Right: percentage split
axes[1].bar(x, monthly_type['p2p_pct'], label='P2P %', color='#5C2D91', alpha=0.85, width=0.6)
axes[1].bar(x, monthly_type['p2m_pct'], bottom=monthly_type['p2p_pct'],
            label='P2M %', color='#0EA5E9', alpha=0.85, width=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(MONTH_SHORT)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_title('P2P vs P2M — Percentage Split')
axes[1].set_ylim(0, 110)
axes[1].axhline(50, color='white', linewidth=1.5, linestyle='--', alpha=0.7)
axes[1].legend(loc='upper left')

# Add P2M % labels on right chart
for i, (p2m, total) in enumerate(zip(monthly_type['p2m_pct'], monthly_type['total'])):
    axes[1].text(i, 102, f'{p2m:.0f}%', ha='center', fontsize=8.5,
                 color='#0EA5E9', fontweight='bold')

fig.suptitle('P2P vs P2M Transaction Split by Month — 2024',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/03_p2p_vs_p2m.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 3 saved ✓")


## Chart 4 — Transaction Volume Heatmap by Hour & Day
**Story:** Peak activity is clearly 9am–12pm and 7pm–10pm.
Weekends show a more even spread across hours — leisure spending pattern.
This is the most visually striking chart in the portfolio.


In [ ]:
# Build hour × day_of_week pivot
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = (
    df_clean.groupby(['day_of_week', 'hour'])
    .size().reset_index(name='count')
)
heatmap_pivot = heatmap_data.pivot(index='day_of_week', columns='hour', values='count').fillna(0)
heatmap_pivot = heatmap_pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(15, 5))
sns.heatmap(
    heatmap_pivot,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.4,
    linecolor='white',
    annot=False,
    fmt='.0f',
    cbar_kws={'label': 'Number of Transactions', 'shrink': 0.8}
)
ax.set_xlabel('Hour of Day (24hr)', labelpad=8)
ax.set_ylabel('')
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=8.5)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.set_title('UPI Transaction Volume — Hour of Day × Day of Week Heatmap (2024)',
             pad=12)

# Mark peak zones with a border
from matplotlib.patches import Rectangle
# Morning peak: hours 9-11 (columns 9-11)
ax.add_patch(Rectangle((9, 0), 3, 7, fill=False, edgecolor='#5C2D91',
                        linewidth=2.5, label='Morning peak'))
# Evening peak: hours 19-21
ax.add_patch(Rectangle((19, 0), 3, 7, fill=False, edgecolor='#0EA5E9',
                        linewidth=2.5, label='Evening peak'))

ax.legend(handles=[
    plt.Line2D([0],[0], color='#5C2D91', linewidth=2.5, label='Morning peak (9–12am)'),
    plt.Line2D([0],[0], color='#0EA5E9', linewidth=2.5, label='Evening peak (7–10pm)'),
], loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/04_hourly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 4 saved ✓")


## Chart 5 — Top 10 Cities by Transaction Volume & Avg Value
**Story:** Mumbai, Delhi and Bengaluru lead in volume. But mid-tier cities like
Hyderabad and Pune show disproportionately high average transaction values —
indicating more premium or business-oriented usage.


In [ ]:
city_stats = (
    df_clean[~df_clean['city'].isin(['Unknown', 'Others'])]
    .groupby('city')
    .agg(txn_count=('transaction_id','count'),
         avg_value=('amount_inr','mean'),
         total_value=('amount_inr','sum'))
    .reset_index()
    .sort_values('txn_count', ascending=False)
    .head(10)
    .sort_values('txn_count', ascending=True)  # flip for horizontal bar
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = plt.cm.YlOrRd(np.linspace(0.3, 0.85, len(city_stats)))

# Left: transaction count
bars1 = axes[0].barh(city_stats['city'], city_stats['txn_count'],
                     color=bar_colors, height=0.6, edgecolor='white')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_title('Top 10 Cities — Transaction Volume')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
for bar, val in zip(bars1, city_stats['txn_count']):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9)
axes[0].set_xlim(0, city_stats['txn_count'].max() * 1.15)

# Right: avg transaction value
bars2 = axes[1].barh(city_stats['city'], city_stats['avg_value'],
                     color=bar_colors, height=0.6, edgecolor='white')
axes[1].set_xlabel('Average Transaction Value (₹)')
axes[1].set_title('Top 10 Cities — Avg Transaction Value')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
for bar, val in zip(bars2, city_stats['avg_value']):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'₹{val:,.0f}', va='center', fontsize=9)
axes[1].set_xlim(0, city_stats['avg_value'].max() * 1.18)

fig.suptitle('UPI Transaction Activity — Top 10 Cities (2024)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/05_top_cities.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 5 saved ✓")


## Chart 6 — Amount Bracket Distribution
**Story:** Validates against real NPCI data — 86% of UPI transactions are under ₹500.
Our synthetic data should closely mirror this. The cumulative line makes the
86% threshold immediately visible.


In [ ]:
bracket_order = ['₹0–100','₹101–500','₹501–2,000',
                  '₹2,001–10,000','₹10,001–1,00,000','> ₹1,00,000']
bracket_stats = (
    df_clean.groupby('amount_bracket')
    .agg(count=('transaction_id','count'),
         total_value=('amount_inr','sum'))
    .reindex(bracket_order)
    .reset_index()
)
bracket_stats['pct'] = bracket_stats['count'] / bracket_stats['count'].sum() * 100
bracket_stats['cumulative_pct'] = bracket_stats['pct'].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 5))
bar_colors = ['#5C2D91','#7C4DB3','#9C6DC5','#BCAAD4','#D4C5E8','#EDE0F5']
bars = ax1.bar(range(len(bracket_stats)), bracket_stats['pct'],
               color=bar_colors, width=0.6, edgecolor='white')
ax1.set_xticks(range(len(bracket_stats)))
ax1.set_xticklabels(bracket_stats['amount_bracket'], rotation=15, ha='right')
ax1.set_ylabel('% of Transactions')
ax1.set_title('Transaction Amount Bracket Distribution — 2024')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))

# Value labels on bars
for bar, pct, cnt in zip(bars, bracket_stats['pct'], bracket_stats['count']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{pct:.1f}%
({cnt:,})', ha='center', va='bottom', fontsize=9)

# Cumulative line
ax2 = ax1.twinx()
ax2.plot(range(len(bracket_stats)), bracket_stats['cumulative_pct'],
         color='#FF6B35', linewidth=2.5, marker='D', markersize=7, label='Cumulative %')
ax2.set_ylabel('Cumulative %', color='#FF6B35')
ax2.tick_params(axis='y', labelcolor='#FF6B35')
ax2.set_ylim(0, 115)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))

# 86% reference line (NPCI real stat)
ax2.axhline(86, color='#FF6B35', linewidth=1.5, linestyle='--', alpha=0.6)
ax2.text(5.4, 87.5, 'NPCI: 86% < ₹500', color='#FF6B35', fontsize=9,
         ha='right', fontweight='bold')

ax2.legend(loc='center right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/06_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
# Print the actual % under ₹500 for validation
under_500 = bracket_stats[bracket_stats['amount_bracket'].isin(['₹0–100','₹101–500'])]['pct'].sum()
print(f"Chart 6 saved ✓")
print(f"Our data: {under_500:.1f}% of transactions are under ₹500  |  NPCI real stat: 86%")


## Chart 7 — Merchant Category Breakdown (P2M only)
**Story:** Grocery & Supermarket dominates P2M spending, followed by
Food & Restaurants. Together they show how UPI has become the default
payment method for daily essentials.


In [ ]:
cat_stats = (
    df_clean[
        (df_clean['transaction_type'] == 'P2M') &
        (~df_clean['merchant_category'].isin(['N/A (P2P)','Uncategorised','Others']))
    ]
    .groupby('merchant_category')
    .agg(txn_count=('transaction_id','count'),
         total_value=('amount_inr','sum'),
         avg_value=('amount_inr','mean'))
    .reset_index()
    .sort_values('txn_count', ascending=True)
)
cat_stats['pct'] = cat_stats['txn_count'] / cat_stats['txn_count'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cmap = plt.cm.get_cmap('YlOrRd', len(cat_stats))
bar_colors = [cmap(i) for i in np.linspace(0.25, 0.9, len(cat_stats))]

# Left: transaction count
bars1 = axes[0].barh(cat_stats['merchant_category'], cat_stats['txn_count'],
                     color=bar_colors, height=0.65, edgecolor='white')
axes[0].set_xlabel('Number of Transactions')
axes[0].set_title('P2M Merchant Categories — Transaction Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
for bar, val, pct in zip(bars1, cat_stats['txn_count'], cat_stats['pct']):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'{val:,}  ({pct:.1f}%)', va='center', fontsize=8.5)
axes[0].set_xlim(0, cat_stats['txn_count'].max() * 1.28)

# Right: avg transaction value
bars2 = axes[1].barh(cat_stats['merchant_category'], cat_stats['avg_value'],
                     color=bar_colors, height=0.65, edgecolor='white')
axes[1].set_xlabel('Average Transaction Value (₹)')
axes[1].set_title('P2M Merchant Categories — Avg Transaction Value')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'₹{v:,.0f}'))
for bar, val in zip(bars2, cat_stats['avg_value']):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                 f'₹{val:,.0f}', va='center', fontsize=8.5)
axes[1].set_xlim(0, cat_stats['avg_value'].max() * 1.22)

fig.suptitle('UPI Merchant Payment (P2M) Category Analysis — 2024',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/07_merchant_categories.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 7 saved ✓")


## Chart 8 — Month-over-Month Growth Rate
**Story:** Q4 shows the strongest growth momentum driven by festive season spending.
February shows a dip (short month effect). Negative growth months are clearly
flagged in red — important for anomaly detection discussions in interviews.


In [ ]:
ms_growth = ms.copy()
ms_growth['mom_growth'] = ms_growth['total_transactions'].pct_change() * 100
ms_growth['mom_value_growth'] = ms_growth['total_value_cr'].pct_change() * 100
ms_growth = ms_growth.dropna(subset=['mom_growth'])

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Top panel: volume growth
colors_vol = ['#E53E3E' if v < 0 else '#5C2D91' for v in ms_growth['mom_growth']]
bars1 = axes[0].bar(range(len(ms_growth)), ms_growth['mom_growth'],
                    color=colors_vol, width=0.6, edgecolor='white')
axes[0].axhline(0, color='#64748B', linewidth=1.2)
axes[0].set_ylabel('MoM Growth (%)')
axes[0].set_title('Month-over-Month Transaction Volume Growth Rate')
for bar, val in zip(bars1, ms_growth['mom_growth']):
    ypos = bar.get_height() + 0.2 if val >= 0 else bar.get_height() - 1.2
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'{val:+.1f}%', ha='center', va='bottom', fontsize=9,
                 color='#E53E3E' if val < 0 else '#5C2D91', fontweight='bold')

# Bottom panel: value growth
colors_val = ['#E53E3E' if v < 0 else '#0EA5E9' for v in ms_growth['mom_value_growth']]
bars2 = axes[1].bar(range(len(ms_growth)), ms_growth['mom_value_growth'],
                    color=colors_val, width=0.6, edgecolor='white')
axes[1].axhline(0, color='#64748B', linewidth=1.2)
axes[1].set_ylabel('MoM Growth (%)')
axes[1].set_title('Month-over-Month Transaction Value Growth Rate')
axes[1].set_xticks(range(len(ms_growth)))
axes[1].set_xticklabels(MONTH_SHORT[1:])  # Skip January (no prior month)
for bar, val in zip(bars2, ms_growth['mom_value_growth']):
    ypos = bar.get_height() + 0.2 if val >= 0 else bar.get_height() - 1.2
    axes[1].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'{val:+.1f}%', ha='center', va='bottom', fontsize=9,
                 color='#E53E3E' if val < 0 else '#0EA5E9', fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#5C2D91', label='Positive growth — Volume'),
                   Patch(facecolor='#0EA5E9', label='Positive growth — Value'),
                   Patch(facecolor='#E53E3E', label='Negative growth')]
axes[0].legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}/08_mom_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart 8 saved ✓")


## Summary — All Charts Generated

In [ ]:
import os

charts = sorted(os.listdir(CHARTS_DIR))
print("=" * 55)
print("EDA COMPLETE — Charts saved")
print("=" * 55)
for c in charts:
    size_kb = os.path.getsize(f'{CHARTS_DIR}/{c}') // 1024
    print(f"  {c:<45} {size_kb:>4} KB")

print()
print("Key findings from this dataset:")
print(f"  Total transactions analysed : {len(df_clean):,}")
print(f"  Date range                  : Jan 2024 – Dec 2024")
print(f"  Peak month (volume)         : {ms.loc[ms['total_transactions'].idxmax(), 'month_name']}")
print(f"  Peak month (value)          : {ms.loc[ms['total_value_cr'].idxmax(), 'month_name']}")
print(f"  Dominant app                : {df_clean['upi_app'].value_counts().index[0]}")
print(f"  Top city                    : {df_clean[~df_clean['city'].isin(['Unknown','Others'])]['city'].value_counts().index[0]}")
print(f"  P2M share (Dec)             : {ms[ms['month']==12]['p2m_share_%'].values[0]:.1f}%")
under_500 = (df_clean['amount_inr'] <= 500).sum() / len(df_clean) * 100
print(f"  Transactions under ₹500     : {under_500:.1f}%  (NPCI real: 86%)")
print()
print("Next step: Power BI Dashboard (Step 5)")


---
**Next:** `dashboard/` — Power BI Dashboard with 6 visuals